# Ballet AI + LSTM 詩意生成（自動訓練版）
> 執行這支筆記本 → 自動訓練 + 儲存模型 + 執行 demo

In [ ]:
# Cell 1: 自動訓練 LSTM 模型（修正縮排）
import torch
import torch.nn as nn
import torch.optim as optim
import random

# === 詞彙表 ===
vocab = {
    "<PAD>":0, "<SOS>":1, "<EOS>":2,
    "head":3, "arm":4, "leg":5, "torso":6,
    "still":7, "slight":8, "medium":9, "fast":10, "spin":11, "high":12,
    "lift":13, "turn":14, "stretch":15, "jump":16,
    "like":17, "a":18, "in":19, "the":20, "her":21, "soul":22,
    "flies":23, "light":24, "poetry":25, "moment":26, "freedom":27,
    "dances":28, "stars":29, "dream":30, "into":31, "eternity":32,
    "feather":33, "wind":34, "takes":35, "flight":36, "becomes":37, "pure":38
}
idx2word = {i:w for w,i in vocab.items()}
vocab_size = len(vocab)

# === 假資料模板 ===
templates = [
    ["<SOS>", "lift", "like", "a", "feather", "in", "the", "wind", "<EOS>"],
    ["<SOS>", "spin", "her", "soul", "flies", "<EOS>"],
    ["<SOS>", "stretch", "becomes", "poetry", "<EOS>"],
    ["<SOS>", "jump", "moment", "of", "freedom", "<EOS>"]
]

def make_data(n=500):
    data = []
    for _ in range(n):
        sent = random.choice(templates)
        if random.random() < 0.3:
            act = random.choice(["still", "slight", "fast", "high"])
            sent = ["<SOS>", act, "like", "a", "feather", "in", "the", "wind", "<EOS>"]
        data.append([vocab.get(w,0) for w in sent])
    return data

train_data = make_data()

開始訓練 LSTM 模型...（約 30 秒）


In [ ]:
# === LSTM 模型（縮排已修正！）===
class LSTMGen(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, 64, padding_idx=0)
        self.lstm = nn.LSTM(64, 128, 2, batch_first=True)
        self.fc = nn.Linear(128, vocab_size)
    
    def forward(self, x, h=None):
        x = self.embed(x)
        o, h = self.lstm(x, h)
        return self.fc(o), h

In [ ]:
# === 訓練設定 ===
device = "cpu"
model = LSTMGen().to(device)
opt = optim.Adam(model.parameters(), 0.005)
loss_fn = nn.CrossEntropyLoss(ignore_index=0)

print("開始訓練 LSTM 模型...（約 30 秒）")
model.train()
for epoch in range(300):
    random.shuffle(train_data)
    total = 0
    for seq in train_data:
        if len(seq) < 2: continue
        x = torch.tensor([seq[:-1]], dtype=torch.long).to(device)
        y = torch.tensor([seq[1:]], dtype=torch.long).to(device)
        opt.zero_grad()
        out, _ = model(x)
        loss = loss_fn(out.view(-1, vocab_size), y.view(-1))
        loss.backward()
        opt.step()
        total += loss.item()
    if (epoch+1) % 100 == 0:
        print(f"Epoch {epoch+1}, Loss: {total/len(train_data):.4f}")

In [ ]:
# === 儲存模型 ===
torch.save(model.state_dict(), "lstm_poem.pth")
print("\n模型已儲存：lstm_poem.pth")
print("現在可以執行 demo 了！")